# CS 691 · Module 08 · Choosing predictions that satisfy constraints

Construct the allowed assignments, select the one with the highest total score, and explain what happens when no complete assignment is possible.

**Before you begin.** You can read model scores and distinguish a soft training penalty from a rule applied to a decision. We now apply rules after scores have been produced.

Ava and Bo are each offered the only copy of C1 for 10 a.m. The desk has promised each a camera at that time. Can both promises be kept as written?

<details><summary>Reveal the answer</summary>

No. Two borrowers would need the same physical camera at the same time. At least one promise or the available inventory would have to change.

</details>

## Separate preference from permission

The desk now has a different task. A model scores how well each available camera suits a borrower's request. These are suitability scores for allocation. The desk also knows how many physical cameras it has. A hard constraint states a condition every acceptable output must satisfy. Here the same camera cannot be assigned to two borrowers at once.

Ava's highest score is 0.9 for C1. Ben's highest score is 0.8 for C1. Choosing each person's favorite independently would promise C1 twice.

**Predict before running the cell.** Would lowering Ben's C1 score from 0.8 to 0.75 remove the conflict?

In [ ]:
scores = {"Ava": {"C1": .9, "C2": .7},
          "Ben": {"C1": .8, "C2": .2}}
favorites = {p: max(row, key=row.get) for p, row in scores.items()}
print(favorites["Ava"], favorites["Ben"])
print("Distinct cameras:", len(set(favorites.values())))

<details><summary>Reveal output and explanation</summary>

```text
C1 C1
Distinct cameras: 1
```

No. He still prefers C1 at 0.75 over C2 at 0.2. Independent highest-score choices still promise C1 twice.

</details>

## Generate possibilities and test the rule

A candidate output now contains two choices, ordered as Ava's camera followed by Ben's. An assignment is feasible if it satisfies every hard constraint. In this first exercise both cameras are available, each borrower must get one, and they must get different cameras. List all pairs and keep the feasible ones.

There are four pairs because each of the two people can initially be matched to either camera. Discard (C1, C1) and (C2, C2). The two remaining pairs can be compared by score.

**Predict before running the cell.** If C2 becomes unavailable, is there any way to assign both borrowers a different camera?

In [ ]:
cameras = ["C1", "C2"]
for ava in cameras:
    for ben in cameras:
        if ava != ben:
            print(ava, ben)

<details><summary>Reveal output and explanation</summary>

```text
C1 C2
C2 C1
```

No. The only pair using the available inventory is (C1, C1), which violates distinctness. The desk must handle the absence of a feasible complete assignment.

</details>

## Compare the complete assignments

Decoding means choosing the output from the model's scores. In constrained decoding we choose among feasible outputs. This desk uses the sum of the two suitability scores as its stated objective. Compare complete assignments because choosing Ava's favorite first can leave Ben with a poor match.

Ava C1 and Ben C2 totals 0.9 + 0.2 = 1.1. Ava C2 and Ben C1 totals 0.7 + 0.8 = 1.5. The second pair wins under this objective, although C2 ranks second for Ava.

**Predict before running the cell.** How could you justify the selected pair without mentioning the code?

In [ ]:
ava = {"C1": .9, "C2": .7}
ben = {"C1": .8, "C2": .2}
pairs = [(a, b) for a in ava for b in ben if a != b]
def total(pair):
    a, b = pair
    return ava[a] + ben[b]
best = max(pairs, key=total)
print(f"Ava {best[0]}, Ben {best[1]}, total {total(best):.1f}")

<details><summary>Reveal output and explanation</summary>

```text
Ava C2, Ben C1, total 1.5
```

Write down the two feasible pairs. Both respect the one-camera limit, and their totals are 1.1 and 1.5. Choosing the larger total proves the second pair is best among the feasible choices.

</details>

## Handle a shortage explicitly

Suppose only C1 is available. If the desk requires both borrowers to get a camera, it should report that no complete assignment exists. If the desk explicitly permits waiting, introduce Wait as an allowed choice with score zero. Both people may wait because a waiting place uses no camera.

With waiting allowed and C1 as the only camera, the feasible totals are 0.9 for Ava borrowing, 0.8 for Ben borrowing, and 0 for both waiting. The decision and its assumptions can be shown to the desk.

**Predict before running the cell.** Tomorrow both requests have the same score for C1. Can these scores and constraints resolve who gets it?

In [ ]:
ava = {"C1": .9, "Wait": 0}
ben = {"C1": .8, "Wait": 0}
pairs = [(a, b) for a in ava for b in ben
         if a == "Wait" or b == "Wait" or a != b]
best = max(pairs, key=lambda pair: ava[pair[0]] + ben[pair[1]])
print(f"Ava {best[0]}, Ben {best[1]}")

<details><summary>Reveal output and explanation</summary>

```text
Ava C1, Ben Wait
```

No. Giving C1 to either borrower has the same total. The desk needs an explicit tie-breaking rule, such as earlier request time, or it can report both equally scoring assignments. Loop order is an accidental policy.

</details>

## Compare both ways to share the cameras

The list of pairs keeps only assignments to different cameras. The total function looks up Ava’s score for her camera and Ben’s score for his, then adds them. Passing that function to max tells Python to compare assignments by their totals.

Replace this line:
ben = {"C1": .8, "C2": .2}
with:
ben = {"C1": .8, "C2": .9}
What will print?

In [ ]:
ava = {"C1": .9, "C2": .7}
ben = {"C1": .8, "C2": .2}
pairs = [(a, b) for a in ava for b in ben if a != b]
def total(pair):
    a, b = pair
    return ava[a] + ben[b]
best = max(pairs, key=total)
print(f"Ava {best[0]}, Ben {best[1]}, total {total(best):.1f}")

<details><summary>Worked answer after your change</summary>

Ava C1, Ben C2, total 1.8

The original winning pair still totals 1.5, but assigning C1 to Ava and C2 to Ben now totals 0.9 + 0.9 = 1.8.

</details>

## Exercise 1

Ava and Ben can request C1 or C2. How many complete pairs are possible before and after applying the distinct-camera rule?

<details><summary>Hint</summary>

Treat (Ava C1, Ben C2) and (Ava C2, Ben C1) as different.

</details>

<details><summary>Worked answer</summary>

Two choices for each of two people give four pairs. Remove the two pairs that reuse a camera. Two distinct-camera pairs remain.

</details>

## Exercise 2

Ava's scores are 0.6 for C1 and 0.5 for C2. Ben's scores are 0.9 for C1 and 0.8 for C2. Both must receive distinct cameras. Is there a unique winner?

<details><summary>Hint</summary>

Calculate both feasible totals before deciding.

</details>

<details><summary>Worked answer</summary>

Ava C1 plus Ben C2 totals 1.4. Ava C2 plus Ben C1 also totals 1.4. Neither the hard rule nor the total breaks this tie.

</details>

## Exercise 3

A colleague replaces the hard rule with a penalty of 0.1 for sharing. Ava and Ben each score C1 at 1.0 and C2 at 0.2. What can the new objective select?

<details><summary>Hint</summary>

Compare the penalized shared-camera total with the best distinct-camera total.

</details>

<details><summary>Worked answer</summary>

Giving both people C1 scores 2.0 − 0.1 = 1.9. Giving them different cameras scores 1.2. The penalized objective prefers an impossible handoff. A hard feasibility check rules it out regardless of scores.

</details>

A score and an inventory rule can help the desk allocate a camera. Once a reservation is recorded, Ava has a new question. Which records will tell her where to collect her camera?

After allocating C2 to Ava, the desk moves C2’s pickup point. Should the old suitability score or the new pickup record determine where Ava goes?

<details><summary>Reveal the answer</summary>

Use the reservation to identify C2, then consult the current pickup record for C2. The suitability score helped allocate a camera and contains no location information.

</details>